# Sprint 4 · Finish and test a recoverable handoff

Linden Facilities needs 24 hourly predictions as one file within 60 seconds. Continue in the same repository and `work/`; the saved experiment plan and selected run carry into later sprints. This is an optional exploration notebook. [Campus lessons](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md) also provide terminal/API routes. Keep one set of work. There is nothing to submit.

Follow [local setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select the project's Python 3.12 kernel, and run setup. Restarting a kernel loses variables, not saved run IDs. Do not reset the project between sections. Independent cells are for your attempt; guided outputs alone do not answer them.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use Python 3.12. Locally run uv sync --locked --python 3.12 and select .venv/bin/python.')
# Hosted exploration is optional. Local setup uses the already installed editable project.
try:
    import linden_mlops
except ImportError:
    if 'google.colab' not in sys.modules:
        raise RuntimeError('Follow docs/setup.md from the repository root, then select its .venv kernel.')
    hosted_root = Path('/content/ml-llm-ops')
    if not hosted_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/richhiey/ml-llm-ops.git', str(hosted_root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(hosted_root), 'pytest==8.4.2'], check=True)
    sys.path.insert(0, str(hosted_root / 'src'))
    import linden_mlops
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate, run_pipeline,
                          export_runs, verify_summary, register, approve, activate, batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
print('Working source:', ROOT)
print('Retained work:', state)
print('Code:', code_revision())
print('Saved decisions:', sorted(progress.data))

## 1 · Scope what still needs proof

Before executing more pipeline work, inspect what exists. Use the saved selection, source diff, contract checks, registration and batch evidence to identify missing proof. Make a compact inventory in one `work/HANDOFF.md`: criterion, existing evidence, gap and next verification command. Keep improving this same runbook, not a series of assignment reports.

| Criterion | Inspect |
|---|---|
| Connected lifecycle | selected run -> artifact -> registered version -> active batch |
| Explained extension | accepted/rejected inputs + unchanged-stage proof |
| Recoverable handoff | exact recipe + portable files + fresh receiving check |

The validated range stage is already a non-trivial extension. Add another feature only if your scope reveals a real missing contract. An optional estimator transfer must preserve interfaces and establish its own comparison; it is not required.

In [ ]:
selection = progress.selected()
print('Selected recipe:', selection)
print('Registry:', progress.data.get('registry'))
print('Promotion:', progress.data.get('promotion'))
print('Existing work:', sorted(p.name for p in state.iterdir()))

In [ ]:
# Create/update state / 'HANDOFF.md' with existing evidence and only the remaining gaps.
# Scope first; do not rebuild a completed pipeline.

## 2 · Connect existing execution evidence

Inspect `cli.py` and `lifecycle.py`: which function owns train/evaluate/log/register/approve/activate/batch? Reuse the selected run and existing registry identity. Check source run equality before claiming connection. The fresh receiving run later supplies a new full execution; do not rerun a baseline just for this inventory.

Exercise one failed candidate with the guard disabled in this same store, keeping the active version unchanged. Registration is permitted; approval must reject. A failure should leave an inspectable candidate without changing serving.

In [ ]:
version = progress.require('registry')['version']
active_before = ws.client.get_model_version_by_alias(MODEL_NAME, 'active')
assert active_before.run_id == selection['run_id']
if not progress.data.get('rejected_candidate'):
    candidate = ws.run(Config(alpha=selection['config']['alpha'], range_guard=False, autolog=True), name='guard-disabled-negative')
    candidate_version = register(ws, candidate['run_id'])
    try:
        approve(ws, candidate_version, 'Deliberate guard-disabled negative test')
    except ValueError as error:
        print('Expected rejection:', error)
    else:
        raise AssertionError('Guard-disabled candidate approved')
    progress.save(rejected_candidate={'run_id': candidate['run_id'], 'version': candidate_version})
assert ws.client.get_model_version_by_alias(MODEL_NAME, 'active').version == active_before.version
print('Preserved active:', active_before.version, active_before.run_id)
print('Rejected candidate:', progress.require('rejected_candidate'))

## 3 · Verify the selected serving path

Run one representative batch if the existing result does not already prove this unchanged input/selection. Inspect its output and completion evidence: 24 finite predictions, same identifiers/order, selected run/version, and whole-job time within 60 seconds. Training/setup time is outside this batch measurement; no endpoint timing or cost claim follows from it.

Then probe a copy with an invalid hour. Preserve the valid output under a separate name. An expected rejection must not be presented as a successful fast batch.

In [ ]:
if not progress.data.get('serving_check'):
    output, serving = batch_predict(ws, output_name='serving-check')
    progress.save(serving_check=serving)
serving = progress.require('serving_check')
output = pd.read_csv(ws.path / 'serving-check.csv')
from linden_mlops.pipeline import DATA
planned = pd.read_csv(DATA / 'batch-v1.csv')
assert output.row_id.tolist() == planned.row_id.tolist()
assert len(output) == 24 and np.isfinite(output.prediction_kwh).all()
assert serving['run_id'] == selection['run_id'] and serving['resolved_version'] == version
assert serving['complete_within_60s']
print(serving)
display(output.head())

In [ ]:
# Create a copy of planned with hour=24, save under state, and try batch_predict.
# Expect ValueError, preserve the valid serving-check files and original input.
# Explain the observed timing's scope.

## 4 · Package the recorded recipe and write instructions

Use [recovery](https://github.com/richhiey/ml-llm-ops/blob/main/docs/recovery.md) as a complete reference. Export the selected validation run's actual source/config/environment/model, rather than substitute default alpha `1`. The export has checksum-verified files; old absolute MLflow URIs do not need to relocate.

Update your existing `HANDOFF.md` with actual input/export paths, working directory, Python/lock commands, expected invariant fields, tolerance and inference acceptance. Include diagnosis for a wrong interpreter, source mismatch, missing export file or existing destination. Another engineer must be able to follow it without notebook variables.

In [ ]:
from linden_mlops.recovery import export_run, verify_bundle
bundle = state / 'handoff-export'
if not bundle.exists():
    export_run(ws, selection['run_id'], bundle)
manifest = verify_bundle(bundle)
assert manifest['run_id'] == selection['run_id']
assert manifest['config'] == selection['config']
print('Portable files:', bundle)
print('Use this absolute export path in receiving commands:', bundle.resolve())

In [ ]:
# Adapt the documented recovery commands to your actual export and observed recipe.
# Keep one state / 'HANDOFF.md'; do not create another report.

## 5 · Execute your runbook in a fresh receiving environment

Run the following sequence in terminals, replacing paths with your actual locations. The source destination and receiving state must not exist. Keep the original store unavailable for the receiving command, preserving it intact elsewhere.

```sh
# Original project:
uv run linden-mlops restore-source --bundle /ABSOLUTE/EXPORT --destination ../linden-received
# New restored source folder:
cd ../linden-received
uv sync --locked --python 3.12
uv run pytest -q
uv run linden-mlops reproduce-export --bundle /ABSOLUTE/EXPORT --destination work/received
```

Read `work/received/recovery-check.json`. Require all invariant checks and `passed=true`, absolute validation RMSE delta <= `1e-6` kWh, intended active version and 24 aligned finite predictions within 60 seconds. New IDs/counters/paths are expected. Source/data/config/partition/environment must match. A new store in the old environment proves less than this test.

If anything diverges, retain the failure, diagnose the first mismatching invariant, fix instructions or restore the recipe, and retry a new destination. Reuse the successful receiving result at the checkpoint; do not repeat it merely for a new page.

In [ ]:
# After executing your runbook, inspect the report and batch from the receiving folder.
# Add actual corrections or mismatches to your existing runbook; no generic reflection required.

## 6 · Harbor checks the handoff

Use the existing receiving run and package. Harbor supplies a copy containing `occupancy=101`. In the receiving environment, test this unseen input with the taught guard and explain the rejecting stage. Preserve valid behavior and original bytes. Check the explained extension and connected lifecycle using the records already made; no second project build is required.

After selection is fixed, observe the reserved final partition once below. Keep the result separate from validation-based promotion and do not retune using it. The main selected validation artifact remains the handoff model. There is nothing to submit.

In [ ]:
if not progress.data.get('final_evaluation'):
    final = ws.run(Config(**selection['config']), name='fixed-selection-final', partition='final')
    progress.save(final_evaluation=final)
final = progress.require('final_evaluation')
print('One reserved final observation:', final['run_id'], final['metrics'])
assert final['config'] == selection['config']
assert ws.client.get_model_version_by_alias(MODEL_NAME, 'active').run_id == selection['run_id']

In [ ]:
# In the receiving environment, probe occupancy=101 using a fresh input copy.
# Explain the actual rejection and connect it to the saved extension.
# Reuse the successful recovery report and batch for your self-check.